# ⭐ Bài tập về nhà 3 — Export YOLO26n sang ONNX, đo latency trên CPU
Lab Ngày 18 · Track 4. Đề: *Export `yolo26n.pt` sang ONNX (`model.export(format="onnx")`), đo latency trên CPU của head
one-to-many + NMS và head one-to-one ở conf 0.001 và 0.25.*

- Chạy trên Colab **runtime CPU** (không GPU), cùng `ultralytics==8.4.171` như notebook chính.
- Cách đo giống ô `bench` ở mục 1C: warm-up 1 lần, rồi lấy trung bình `Results.speed` của 30 lần chạy trên `bus.jpg`.
- Đo thêm bản PyTorch trên cùng CPU để có mốc so sánh với ONNX Runtime.

In [1]:
%pip install -q "ultralytics==8.4.171" onnx onnxslim onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.5/244.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21

In [2]:
import os, platform, shutil
from pathlib import Path

import onnxruntime as ort
import pandas as pd
import torch
import ultralytics
from ultralytics import YOLO

IMG_BUS = str(Path(ultralytics.__file__).parent / "assets" / "bus.jpg")
cpu = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), platform.processor())
print(f"torch {torch.__version__} | ultralytics {ultralytics.__version__} | onnxruntime {ort.__version__}")
print(f"CPU: {cpu} · {os.cpu_count()} luồng · torch.get_num_threads() = {torch.get_num_threads()} · CUDA: {torch.cuda.is_available()}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
torch 2.11.0+cpu | ultralytics 8.4.171 | onnxruntime 1.30.0
CPU: AMD EPYC 7B12 · 2 luồng · torch.get_num_threads() = 1 · CUDA: False


## 1. Export hai head sang ONNX
Ở `ultralytics 8.4.171`, tham số `nms` của `export` chọn head:
- `nms=None` (mặc định của `model.export(format="onnx")`): giữ head **one-to-many**, output thô `(1, 84, 8400)`, NMS chạy **ngoài** graph khi suy luận;
- `nms=False`: chọn head **one-to-one**, NMS-free, output `(1, 300, 6)` đã là danh sách box cuối cùng.

In [3]:
ONNX = {}
for head, kw in {"one-to-many": {}, "one-to-one": {"nms": False}}.items():
    f = YOLO("yolo26n.pt").export(format="onnx", imgsz=640, device="cpu", **kw)
    ONNX[head] = shutil.move(f, f"yolo26n_{head}.onnx")

for head, path in ONNX.items():
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    print(f"{head:12s} {path}: input {sess.get_inputs()[0].shape} → output {sess.get_outputs()[0].shape} · "
          f"{Path(path).stat().st_size / 1e6:.1f} MB · provider {sess.get_providers()[0]}")

Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
YOLO26n summary (fused): 120 layers, 2,408,932 parameters, 0 gradients, 5.5 GFLOPs

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.3 MB)

ONNX: starting export with onnx 1.23.1 opset 18...
ONNX: slimming with onnxslim 0.1.97...
ONNX: export success ✅ 8.4s, saved as 'yolo26n.onnx' (9.5 MB)

Export complete (9.1s)
Results saved to /content/yolo26n.onnx
Predict:         yolo predict task=detect model=yolo26n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolo26n.onnx imgsz=640   
Visualize:       https://netron.app
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.5 GFLOPs

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.3 MB)

ONNX: starting export with onnx 1.23.1 opset 18...
ONNX

## 2. Đo latency trên CPU
Bốn cấu hình của đề (2 head × 2 mức conf) cho ONNX Runtime, cộng bốn cấu hình tương ứng của PyTorch.
Dùng **object riêng cho mỗi head**, như notebook chính: một object PyTorch đã gọi với `nms=False` sẽ giữ head one-to-one.
Model ONNX có input cố định 640×640 nên `bus.jpg` được letterbox thành 640×640; bản PyTorch letterbox thành 640×480.

In [4]:
def bench(model, n=30, **kw):
    """Thời gian trung bình (ms) của từng giai đoạn trên bus.jpg, lấy từ Results.speed."""
    model.predict(IMG_BUS, device="cpu", verbose=False, **kw)                 # warm-up
    acc = {"preprocess": 0.0, "inference": 0.0, "postprocess": 0.0}
    for _ in range(n):
        r = model.predict(IMG_BUS, device="cpu", verbose=False, **kw)[0]
        for k in acc:
            acc[k] += r.speed[k] / n
    return {f"{k} (ms)": round(v, 2) for k, v in acc.items()} | {"tổng (ms)": round(sum(acc.values()), 2), "số box": len(r.boxes)}


CONFIGS = {
    ("ONNX", "one-to-many + NMS"): (YOLO(ONNX["one-to-many"], task="detect"), {"iou": 0.7}),
    ("ONNX", "one-to-one NMS-free"): (YOLO(ONNX["one-to-one"], task="detect"), {}),
    ("PyTorch", "one-to-many + NMS"): (YOLO("yolo26n.pt"), {"iou": 0.7}),
    ("PyTorch", "one-to-one NMS-free"): (YOLO("yolo26n.pt"), {"nms": False}),
}
LATENCY_CPU = [{"Backend": backend, "Head": head, "conf": conf} | bench(model, conf=conf, **kw)
               for (backend, head), (model, kw) in CONFIGS.items() for conf in (0.25, 0.001)]
df = pd.DataFrame(LATENCY_CPU)
print(f"Latency YOLO26n trên CPU ({cpu}), trung bình 30 lần:")
df

Loading yolo26n_one-to-many.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.30.0 with CPUExecutionProvider
Loading yolo26n_one-to-one.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.30.0 with CPUExecutionProvider
Latency YOLO26n trên CPU (AMD EPYC 7B12), trung bình 30 lần:


,Backend,Head,conf,preprocess (ms),inference (ms),postprocess (ms),tổng (ms),số box
0,ONNX,one-to-many + NMS,0.250,4.58,99.62,1.66,105.86,5
1,ONNX,one-to-many + NMS,0.001,3.67,89.60,1.94,95.21,186
2,ONNX,one-to-one NMS-free,0.250,3.71,89.79,0.47,93.97,5
3,ONNX,one-to-one NMS-free,0.001,3.69,88.36,0.51,92.56,177
4,PyTorch,one-to-many + NMS,0.250,3.70,99.67,1.14,104.52,5
5,PyTorch,one-to-many + NMS,0.001,3.04,89.63,1.55,94.21,203
6,PyTorch,one-to-one NMS-free,0.250,3.26,92.00,0.33,95.58,5
7,PyTorch,one-to-one NMS-free,0.001,3.46,98.02,0.35,101.83,204


## 3. Riêng phần hậu xử lý: NMS tốn bao nhiêu trên CPU?

In [5]:
key = ["Backend", "conf"]
o2m = df[df.Head.str.startswith("one-to-many")].set_index(key)
o2o = df[df.Head.str.startswith("one-to-one")].set_index(key)
cmp = pd.DataFrame({
    "box sau NMS (one-to-many)": o2m["số box"],
    "postprocess one-to-many + NMS (ms)": o2m["postprocess (ms)"],
    "postprocess one-to-one (ms)": o2o["postprocess (ms)"],
    "postprocess chậm hơn (lần)": (o2m["postprocess (ms)"] / o2o["postprocess (ms)"]).round(1),
    "tổng one-to-many − one-to-one (ms)": (o2m["tổng (ms)"] - o2o["tổng (ms)"]).round(2),
})
cmp

box sau NMS (one-to-many)  postprocess one-to-many + NMS (ms)  \
Backend conf                                                                   
ONNX    0.250                          5                                1.66   
        0.001                        186                                1.94   
PyTorch 0.250                          5                                1.14   
        0.001                        203                                1.55   

               postprocess one-to-one (ms)  postprocess chậm hơn (lần)  \
Backend conf                                                             
ONNX    0.250                         0.47                         3.5   
        0.001                         0.51                         3.8   
PyTorch 0.250                         0.33                         3.5   
        0.001                         0.35                         4.4   

               tổng one-to-many − one-to-one (ms)  
Backend conf                                       
ONNX    0.250                               11.89  
        0.001                                2.65  
PyTorch 0.250                                8.94  
        0.001                               -7.62

## 4. Đo lại chắc hơn: 3 vòng xen kẽ, lấy trung vị
Cột inference ở mục 2 là trung bình 30 lần liên tiếp trên CPU dùng chung của Colab, nên dao động giữa các lần đo (vài ms tới cả chục ms, kể cả giữa hai cấu hình chạy **cùng một file ONNX**, chỉ khác conf) — lớn hơn hẳn phần chênh do NMS. Vì vậy cột tổng ở mục 2–3 chưa đủ để so sánh tổng thời gian.
Đo lại: warm-up 3 lần mỗi cấu hình, chạy **3 vòng xen kẽ** qua cả 8 cấu hình × 30 lần (90 mẫu mỗi cấu hình) và lấy **trung vị** từng giai đoạn; kèm p90 của tổng thời gian.

In [6]:
import numpy as np
from IPython.display import display

STAGES = ("preprocess", "inference", "postprocess")
ROUNDS, N = 3, 30
keys = [(backend, head, conf) for (backend, head) in CONFIGS for conf in (0.25, 0.001)]
samples, nbox = {k: [] for k in keys}, {}
for backend, head, conf in keys:                                  # warm-up 3 lần mỗi cấu hình
    model, kw = CONFIGS[(backend, head)]
    for _ in range(3):
        model.predict(IMG_BUS, device="cpu", verbose=False, conf=conf, **kw)
for _ in range(ROUNDS):                                           # mỗi vòng đi qua cả 8 cấu hình → dao động theo thời gian chia đều
    for backend, head, conf in keys:
        model, kw = CONFIGS[(backend, head)]
        for _ in range(N):
            r = model.predict(IMG_BUS, device="cpu", verbose=False, conf=conf, **kw)[0]
            samples[(backend, head, conf)].append([r.speed[s] for s in STAGES])
        nbox[(backend, head, conf)] = len(r.boxes)

rows = []
for key in keys:
    arr = np.array(samples[key])                                  # (90, 3) ms
    tot = arr.sum(1)
    rows.append(dict(zip(("Backend", "Head", "conf"), key))
                | {f"{s} (ms)": round(float(np.median(arr[:, i])), 2) for i, s in enumerate(STAGES)}
                | {"tổng trung vị (ms)": round(float(np.median(tot)), 2), "tổng p90 (ms)": round(float(np.percentile(tot, 90)), 2),
                   "số box": nbox[key]})
df_med = pd.DataFrame(rows)
print(f"Trung vị của {ROUNDS} × {N} = {ROUNDS * N} lần đo mỗi cấu hình trên CPU ({cpu}):")
display(df_med)

med = df_med.set_index(["Backend", "conf"])
o2m, o2o = med[med.Head.str.startswith("one-to-many")], med[med.Head.str.startswith("one-to-one")]
nms_ms = (o2m["postprocess (ms)"] - o2o["postprocess (ms)"]).round(2)
print("NMS trên CPU: one-to-many + NMS so với one-to-one, theo trung vị")
pd.DataFrame({"box sau NMS": o2m["số box"], "postprocess one-to-many + NMS (ms)": o2m["postprocess (ms)"],
              "postprocess one-to-one (ms)": o2o["postprocess (ms)"], "NMS thêm (ms)": nms_ms,
              "tổng one-to-many (ms)": o2m["tổng trung vị (ms)"], "tổng one-to-one (ms)": o2o["tổng trung vị (ms)"],
              "NMS / tổng one-to-many (%)": (100 * nms_ms / o2m["tổng trung vị (ms)"]).round(1)})

Trung vị của 3 × 30 = 90 lần đo mỗi cấu hình trên CPU (AMD EPYC 7B12):


,Backend,Head,conf,preprocess (ms),inference (ms),postprocess (ms),tổng trung vị (ms),tổng p90 (ms),số box
0,ONNX,one-to-many + NMS,0.250,3.48,91.99,1.40,97.88,119.75,5
1,ONNX,one-to-many + NMS,0.001,3.32,87.15,1.82,92.40,96.26,186
2,ONNX,one-to-one NMS-free,0.250,3.35,87.82,0.47,91.70,95.51,5
3,ONNX,one-to-one NMS-free,0.001,3.34,87.81,0.50,91.82,99.41,177
4,PyTorch,one-to-many + NMS,0.250,2.93,89.62,1.10,93.82,126.96,5
5,PyTorch,one-to-many + NMS,0.001,2.81,87.17,1.57,91.81,97.33,203
6,PyTorch,one-to-one NMS-free,0.250,2.85,87.94,0.29,91.28,96.87,5
7,PyTorch,one-to-one NMS-free,0.001,2.87,88.91,0.31,92.20,123.48,204


NMS trên CPU: one-to-many + NMS so với one-to-one, theo trung vị


box sau NMS  postprocess one-to-many + NMS (ms)  \
Backend conf                                                     
ONNX    0.250            5                                1.40   
        0.001          186                                1.82   
PyTorch 0.250            5                                1.10   
        0.001          203                                1.57   

               postprocess one-to-one (ms)  NMS thêm (ms)  \
Backend conf                                                
ONNX    0.250                         0.47           0.93   
        0.001                         0.50           1.32   
PyTorch 0.250                         0.29           0.81   
        0.001                         0.31           1.26   

               tổng one-to-many (ms)  tổng one-to-one (ms)  \
Backend conf                                                 
ONNX    0.250                  97.88                 91.70   
        0.001                  92.40                 91.82   
PyTorch 0.250                  93.82                 91.28   
        0.001                  91.81                 92.20   

               NMS / tổng one-to-many (%)  
Backend conf                               
ONNX    0.250                         1.0  
        0.001                         1.4  
PyTorch 0.250                         0.9  
        0.001                         1.4